In [1]:
from transformers import AutoImageProcessor, ConvNextV2ForImageClassification, Trainer, TrainingArguments
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torch.utils.data import DataLoader
import torch
from datasets import load_dataset
from transformers import default_data_collator
import numpy as np

In [2]:
import multiprocessing as mp

# Set start method to 'spawn' or 'forkserver'
mp.set_start_method('spawn', force=True)

In [3]:
from transformers import AutoImageProcessor, ConvNextV2ForImageClassification
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim
import os

In [4]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split


torch.manual_seed(42)
np.random.seed(42)

In [5]:
from datasets import load_from_disk
train_dataset = load_from_disk('train_dataset')
test_dataset = load_from_disk('val_dataset')

Loading dataset from disk:   0%|          | 0/26 [00:00<?, ?it/s]

In [6]:
train_dataset = train_dataset.rename_column("image", "pixel_values")
test_dataset = test_dataset.rename_column("image", "pixel_values")

In [7]:
print(train_dataset.column_names)


print(train_dataset['label'][:5])

['pixel_values', 'label']
['NORMAL', 'CATARACT', 'CATARACT', 'CATARACT', 'CATARACT']


In [8]:
import torch
from transformers import ConvNextV2ForImageClassification, AutoImageProcessor, Trainer, TrainingArguments
from datasets import load_dataset, load_metric
import numpy as np

In [9]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [10]:
import torch
from transformers import ViTHybridForImageClassification, Trainer, TrainingArguments
from datasets import load_dataset, load_metric
import numpy as np
import wandb


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [11]:
model_name = "google/vit-hybrid-base-bit-384"
model = ViTHybridForImageClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)

Some weights of ViTHybridForImageClassification were not initialized from the model checkpoint at google/vit-hybrid-base-bit-384 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
model.to(device)

ViTHybridForImageClassification(
  (vit): ViTHybridModel(
    (embeddings): ViTHybridEmbeddings(
      (patch_embeddings): ViTHybridPatchEmbeddings(
        (backbone): BitBackbone(
          (bit): BitModel(
            (embedder): BitEmbeddings(
              (convolution): WeightStandardizedConv2d(
                3, 64, kernel_size=(7, 7), stride=(2, 2), bias=False
                (pad): DynamicPad2d()
              )
              (pooler): BitMaxPool2d(
                kernel_size=(3, 3), stride=(2, 2), padding=(0, 0), dilation=(1, 1), ceil_mode=False
                (pad): DynamicPad2d()
              )
              (pad): Identity()
              (norm): BitGroupNormActivation(
                32, 64, eps=1e-05, affine=True
                (activation): ReLU()
              )
            )
            (encoder): BitEncoder(
              (stages): ModuleList(
                (0): BitStage(
                  (layers): Sequential(
                    (0): BitBottleneckLayer(
   

In [13]:
from transformers import  Trainer, TrainingArguments, DefaultDataCollator
from sklearn.preprocessing import LabelEncoder

In [19]:
label_encoder = LabelEncoder()

# Assuming train_dataset and test_dataset are DatasetDict objects
train_labels = train_dataset['label']
test_labels = test_dataset['label']

# Fit the label encoder and transform labels
label_encoder.fit(train_labels + test_labels)

def encode_labels(examples):
    examples['label'] = label_encoder.transform(examples['label'])
    return examples

train_dataset = train_dataset.map(encode_labels, batched=True)
test_dataset = test_dataset.map(encode_labels, batched=True)

Map:   0%|          | 0/7203 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [20]:
metric = load_metric("accuracy")
fp16=True,
def compute_metrics(p):
    pred, labels = p
    pred = pred.argmax(axis=1)
    accuracy = metric.compute(predictions=pred, references=labels)
    return accuracy


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  
    learning_rate=1e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=35,
    run_name="ViTHybrid-binary-classification"  
)


data_collator = DefaultDataCollator()


train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=training_args.per_device_train_batch_size,
    shuffle=True,
    num_workers=4,  
    pin_memory=True,  
    collate_fn=data_collator,
#     prefetch_factor=6
)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=training_args.per_device_eval_batch_size,
    shuffle=False,
    num_workers=4,  
    pin_memory=True,  
    collate_fn=data_collator,

)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  
    eval_dataset=test_dataset,    
    compute_metrics=compute_metrics
)


trainer.train()


results = trainer.evaluate()
print(results)

C:\Users\aruna\AppData\Local\Temp\ipykernel_41940\2772807844.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("accuracy")
c:\Users\aruna\anaconda3\envs\pytorchgpu\lib\site-packages\datasets\load.py:759: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.2/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for

  0%|          | 0/1356 [00:00<?, ?it/s]

c:\Users\aruna\anaconda3\envs\pytorchgpu\lib\site-packages\transformers\models\vit_hybrid\modeling_vit_hybrid.py:266: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  context_layer = torch.nn.functional.scaled_dot_product_attention(


{'loss': 0.8307, 'grad_norm': 1.2940119504928589, 'learning_rate': 9.741887905604721e-05, 'epoch': 0.15}
{'loss': 0.721, 'grad_norm': 0.7565146088600159, 'learning_rate': 9.48377581120944e-05, 'epoch': 0.31}
{'loss': 0.7065, 'grad_norm': 1.3819377422332764, 'learning_rate': 9.22566371681416e-05, 'epoch': 0.46}
{'loss': 0.6968, 'grad_norm': 2.486016273498535, 'learning_rate': 8.96755162241888e-05, 'epoch': 0.62}
{'loss': 0.6976, 'grad_norm': 0.03812867030501366, 'learning_rate': 8.7094395280236e-05, 'epoch': 0.77}
{'loss': 0.6959, 'grad_norm': 1.201185703277588, 'learning_rate': 8.451327433628319e-05, 'epoch': 0.93}


  0%|          | 0/29 [00:00<?, ?it/s]

...

{'eval_loss': 0.692954957485199, 'eval_accuracy': 0.5411111111111111, 'eval_runtime': 187.7324, 'eval_samples_per_second': 4.794, 'eval_steps_per_second': 0.154, 'epoch': 2.0}
{'loss': 0.6943, 'grad_norm': 2.6815004348754883, 'learning_rate': 6.644542772861357e-05, 'epoch': 2.01}
{'loss': 0.6893, 'grad_norm': 0.6084604859352112, 'learning_rate': 6.386430678466077e-05, 'epoch': 2.17}
{'loss': 0.6939, 'grad_norm': 1.5619686841964722, 'learning_rate': 6.128318584070796e-05, 'epoch': 2.32}
{'loss': 0.7006, 'grad_norm': 2.78657865524292, 'learning_rate': 5.870206489675516e-05, 'epoch': 2.48}
{'loss': 0.6883, 'grad_norm': 0.44400107860565186, 'learning_rate': 5.612094395280236e-05, 'epoch': 2.63}
{'loss': 0.6963, 'grad_norm': 0.9183669090270996, 'learning_rate': 5.3539823008849565e-05, 'epoch': 2.79}
{'loss': 0.6939, 'grad_norm': 1.4709893465042114, 'learning_rate': 5.095870206489676e-05, 'epoch': 2.94}


  0%|          | 0/29 [00:00<?, ?it/s]

{'eval_loss': 0.6907315850257874, 'eval_accuracy': 0.5411111111111111, 'eval_runtime': 193.0341, 'eval_samples_per_second': 4.662, 'eval_steps_per_second': 0.15, 'epoch': 3.0}
{'loss': 0.692, 'grad_norm': 1.9265363216400146, 'learning_rate': 4.8377581120943956e-05, 'epoch': 3.1}
{'loss': 0.7027, 'grad_norm': 2.23193359375, 'learning_rate': 4.579646017699115e-05, 'epoch': 3.25}
{'loss': 0.6954, 'grad_norm': 1.8353731632232666, 'learning_rate': 4.321533923303835e-05, 'epoch': 3.41}
{'loss': 0.6891, 'grad_norm': 0.16250558197498322, 'learning_rate': 4.063421828908555e-05, 'epoch': 3.56}
{'loss': 0.6957, 'grad_norm': 1.8799384832382202, 'learning_rate': 3.8053097345132744e-05, 'epoch': 3.72}
{'loss': 0.6936, 'grad_norm': 0.5444175004959106, 'learning_rate': 3.547197640117994e-05, 'epoch': 3.87}


  0%|          | 0/29 [00:00<?, ?it/s]

{'eval_loss': 0.6900184750556946, 'eval_accuracy': 0.5411111111111111, 'eval_runtime': 215.5572, 'eval_samples_per_second': 4.175, 'eval_steps_per_second': 0.135, 'epoch': 4.0}
{'loss': 0.6974, 'grad_norm': 0.5313680768013, 'learning_rate': 3.2890855457227135e-05, 'epoch': 4.03}
{'loss': 0.6926, 'grad_norm': 0.15865924954414368, 'learning_rate': 3.030973451327434e-05, 'epoch': 4.18}
{'loss': 0.7018, 'grad_norm': 0.023751765489578247, 'learning_rate': 2.7728613569321537e-05, 'epoch': 4.34}
{'loss': 0.6914, 'grad_norm': 0.7108350992202759, 'learning_rate': 2.5147492625368735e-05, 'epoch': 4.49}
{'loss': 0.6928, 'grad_norm': 0.23721739649772644, 'learning_rate': 2.2566371681415928e-05, 'epoch': 4.65}
{'loss': 0.6848, 'grad_norm': 0.250455766916275, 'learning_rate': 1.9985250737463127e-05, 'epoch': 4.8}
